In [0]:
# Load all CSV files from /Volumes/workspace/default/phase5
base_path = "/Volumes/workspace/default/phase5/"

# Load each dataset
customers = spark.read.csv(f"{base_path}olist_customers_dataset.csv", header=True, inferSchema=True)
geolocation = spark.read.csv(f"{base_path}olist_geolocation_dataset.csv", header=True, inferSchema=True)
order_items = spark.read.csv(f"{base_path}olist_order_items_dataset.csv", header=True, inferSchema=True)
payments = spark.read.csv(f"{base_path}olist_order_payments_dataset.csv", header=True, inferSchema=True)
reviews = spark.read.csv(f"{base_path}olist_order_reviews_dataset.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{base_path}olist_orders_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{base_path}olist_products_dataset.csv", header=True, inferSchema=True)
sellers = spark.read.csv(f"{base_path}olist_sellers_dataset.csv", header=True, inferSchema=True)
category_translation = spark.read.csv(f"{base_path}product_category_name_translation.csv", header=True, inferSchema=True)

print("Loaded all datasets:")
print(f"  Customers: {customers.count():,} rows")
print(f"  Orders: {orders.count():,} rows")
print(f"  Order Items: {order_items.count():,} rows")
print(f"  Products: {products.count():,} rows")
print(f"  Sellers: {sellers.count():,} rows")
print(f"  Payments: {payments.count():,} rows")
print(f"  Reviews: {reviews.count():,} rows")
print(f"  Geolocation: {geolocation.count():,} rows")
print(f"  Category Translation: {category_translation.count():,} rows")

Loaded all datasets:
  Customers: 99,441 rows
  Orders: 99,441 rows
  Order Items: 112,650 rows
  Products: 32,951 rows
  Sellers: 3,095 rows
  Payments: 103,886 rows
  Reviews: 104,162 rows
  Geolocation: 1,000,163 rows
  Category Translation: 71 rows


In [0]:
# Import all required PySpark functions
from pyspark.sql.functions import (
    col, sum, count, countDistinct, when, to_date, 
    rank, dense_rank, row_number
)
from pyspark.sql.window import Window

print("Imports complete")

Imports complete


In [0]:
# Display schemas and samples of key datasets
print("=== CUSTOMERS ===")
customers.printSchema()
customers.show(3, truncate=False)

print("\n=== ORDERS ===")
orders.printSchema()
orders.show(3, truncate=False)

print("\n=== ORDER ITEMS ===")
order_items.printSchema()
order_items.show(3, truncate=False)

print("\n=== PRODUCTS ===")
products.printSchema()
products.show(3, truncate=False)

=== CUSTOMERS ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151              

In [0]:
# Task 1: Top 3 Customers per City
# Calculate total spend per customer, then rank within each city

# Join: order_items -> orders -> customers
customer_orders = order_items \
    .join(orders, "order_id") \
    .join(customers, "customer_id")

# Calculate total spend per customer per city
customer_spend = customer_orders \
    .groupBy("customer_city", "customer_id") \
    .agg(sum("price").alias("total_spend"))

# Rank customers within each city using window function
window_spec = Window.partitionBy("customer_city").orderBy(col("total_spend").desc())
ranked_customers = customer_spend.withColumn("rank", rank().over(window_spec))

# Filter top 3 per city
task1_result = ranked_customers \
    .filter(col("rank") <= 3) \
    .select(
        col("customer_city").alias("city"),
        "customer_id",
        "total_spend",
        "rank"
    ) \
    .orderBy("city", "rank")

print("=== Task 1: Top 3 Customers per City ===")
print(f"Output columns: city, customer_id, total_spend, rank")
print(f"Total results: {task1_result.count():,} rows\n")
display(task1_result.limit(50))

=== Task 1: Top 3 Customers per City ===
Output columns: city, customer_id, total_spend, rank
Total results: 9,401 rows



city,customer_id,total_spend,rank
abadia dos dourados,9e01f714a2b3b8962c222cf2b74c20dc,199.0,1
abadia dos dourados,a23e3f9a2b656b23b7e52075964b42cd,120.0,2
abadia dos dourados,f11eb8f0b8b87510a93e3e1aa10b0ade,39.9,3
abadiania,576d71ddb21b21763cfedce73b902180,949.99,1
abaete,d47c8bb51df6f716e196ecd6cd5d2c09,449.0,1
abaete,ff0d62f8be4c098e6306f39bc6ebded4,225.9,2
abaete,5371894984937a27cf40c7d20699a786,208.9,3
abaetetuba,c7eb06383ae604616cb4c9d36fe6745e,1500.0,1
abaetetuba,367fd22f1e994de87cf447d29c167a1f,797.6,2
abaetetuba,7727e2cc9ec428ad3173cc812ce1781b,580.27,3


In [0]:
# Task 2: Running Total of Sales
# Calculate daily sales and cumulative (running) total using window function

# Join order_items with orders to get purchase date and price
sales_with_dates = order_items \
    .join(orders, "order_id") \
    .select(
        to_date("order_purchase_timestamp").alias("date"),
        "price"
    )

# Calculate daily sales
daily_sales = sales_with_dates \
    .groupBy("date") \
    .agg(sum("price").alias("daily_sales")) \
    .orderBy("date")

# Window specification for running total: all rows from start to current row
window_spec = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate running total
task2_result = daily_sales \
    .withColumn("running_total", sum("daily_sales").over(window_spec)) \
    .select("date", "daily_sales", "running_total")

print("=== Task 2: Running Total of Sales ===")
print(f"Output columns: date, daily_sales, running_total")
print(f"Total days: {task2_result.count():,}\n")
display(task2_result)

=== Task 2: Running Total of Sales ===
Output columns: date, daily_sales, running_total


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total days: 616



date,daily_sales,running_total
2016-09-04,72.89,72.89
2016-09-05,59.5,132.39
2016-09-15,134.97,267.36
2016-10-02,100.0,367.36
2016-10-03,463.48,830.84
2016-10-04,9940.96,10771.8
2016-10-05,8343.25,19115.05
2016-10-06,7960.51,27075.559999999998
2016-10-07,7228.05,34303.61
2016-10-08,8441.849999999999,42745.46


In [0]:
# Task 3: Top Products per Category
# Aggregate sales per product, join with category, then rank using DENSE_RANK

# Calculate total sales per product
product_sales = order_items \
    .groupBy("product_id") \
    .agg(sum("price").alias("total_sales"))

# Join with products to get category information
product_with_category = product_sales \
    .join(products, "product_id") \
    .select(
        "product_category_name",
        "product_id",
        "total_sales"
    )

# Rank products within each category using DENSE_RANK
window_spec = Window.partitionBy("product_category_name").orderBy(col("total_sales").desc())
ranked_products = product_with_category \
    .withColumn("rank", dense_rank().over(window_spec))

# Filter top 3 products per category
task3_result = ranked_products \
    .filter(col("rank") <= 3) \
    .select(
        col("product_category_name").alias("category"),
        "product_id",
        "total_sales",
        "rank"
    ) \
    .orderBy("category", "rank")

print("=== Task 3: Top Products per Category ===")
print(f"Output columns: category, product_id, total_sales, rank")
print(f"Total results: {task3_result.count():,} rows\n")
display(task3_result.limit(50))

=== Task 3: Top Products per Category ===
Output columns: category, product_id, total_sales, rank
Total results: 221 rows



category,product_id,total_sales,rank
null,5a848e4ab52fd5445cdc07aab1c40e48,24229.029999999984,1
null,eed5cbd74fac3bd79b7c7ec95fa7507d,9945.0,2
null,b1d207586fca400a2370d50a9ba1da98,7152.0,3
agro_industria_e_comercio,11250b0d4b709fee92441c5f34122aed,9111.0,1
agro_industria_e_comercio,423a6644f0aa529e8828ff1f91003690,8043.0,2
agro_industria_e_comercio,672e757f331900b9deea127a2a7b79fd,6885.0,3
alimentos,73326828aa5efe1ba096223de496f596,4461.0199999999995,1
alimentos,89321f94e35fc6d7903d36f74e351d40,3375.709999999999,2
alimentos,ed2067a9c1f79553088a3c67b99a9f97,3187.5,3
alimentos_bebidas,992197904e1d4f0bf3994652373188e4,2586.72,1


In [0]:
# Task 4: Customer Lifetime Value
# Calculate total spend per customer across all orders

# Join order_items with orders to get customer_id
customer_purchases = order_items \
    .join(orders, "order_id") \
    .select("customer_id", "price")

# Calculate total spend per customer
task4_result = customer_purchases \
    .groupBy("customer_id") \
    .agg(sum("price").alias("total_spend")) \
    .orderBy(col("total_spend").desc())

print("=== Task 4: Customer Lifetime Value ===")
print(f"Output columns: customer_id, total_spend")
print(f"Total customers: {task4_result.count():,}\n")
display(task4_result.limit(50))

=== Task 4: Customer Lifetime Value ===
Output columns: customer_id, total_spend
Total customers: 98,666



customer_id,total_spend
1617b1357756262bfa56ab541c47bc16,13440.0
ec5b2ba62e574342386871631fafd3fc,7160.0
c6e2731c5b391845f6800c97401a43a9,6735.0
f48d464a0baaea338cb25f816991ab1f,6729.0
3fd6777bbce08a352fddd04e4a7cc8f6,6499.0
05455dfa7cd02f13d132aa7a6a9729c6,5934.6
df55c14d1476a9a3467f131269c2477f,4799.0
24bbf5fd2f2e1b359ee7de94defc4a15,4690.0
e0a2412720e9ea4f26c1ac985f6a7358,4599.9
3d979689f636322c62418b6346b1c6d2,4590.0


In [0]:
# Task 5: Customer Segmentation
# Apply logic: >10000 → Gold, 5000-10000 → Silver, <5000 → Bronze

# Calculate total spend per customer
customer_lifetime = order_items \
    .join(orders, "order_id") \
    .groupBy("customer_id") \
    .agg(sum("price").alias("total_spend"))

# Apply segmentation logic
customer_segments = customer_lifetime.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
).select("customer_id", "total_spend", "segment")

print("=== Task 5a: Customer Segmentation (Individual Records) ===")
print(f"Output columns: customer_id, total_spend, segment")
display(customer_segments.orderBy(col("total_spend").desc()).limit(50))

# Count customers per segment
segment_counts = customer_segments \
    .groupBy("segment") \
    .agg(count("customer_id").alias("customer_count")) \
    .orderBy("segment")

print("\n=== Task 5b: Customers per Segment ===")
display(segment_counts)

=== Task 5a: Customer Segmentation (Individual Records) ===
Output columns: customer_id, total_spend, segment


customer_id,total_spend,segment
1617b1357756262bfa56ab541c47bc16,13440.0,Gold
ec5b2ba62e574342386871631fafd3fc,7160.0,Silver
c6e2731c5b391845f6800c97401a43a9,6735.0,Silver
f48d464a0baaea338cb25f816991ab1f,6729.0,Silver
3fd6777bbce08a352fddd04e4a7cc8f6,6499.0,Silver
05455dfa7cd02f13d132aa7a6a9729c6,5934.6,Silver
df55c14d1476a9a3467f131269c2477f,4799.0,Bronze
24bbf5fd2f2e1b359ee7de94defc4a15,4690.0,Bronze
e0a2412720e9ea4f26c1ac985f6a7358,4599.9,Bronze
3d979689f636322c62418b6346b1c6d2,4590.0,Bronze



=== Task 5b: Customers per Segment ===


segment,customer_count
Bronze,98660
Gold,1
Silver,5


In [0]:
# Task 6: Final Reporting Table
# Combine: customer_id, city, total_spend, segment, total_orders

# Join all necessary tables
customer_data = order_items \
    .join(orders, "order_id") \
    .join(customers, "customer_id")

# Aggregate metrics per customer
customer_summary = customer_data \
    .groupBy("customer_id", "customer_city") \
    .agg(
        sum("price").alias("total_spend"),
        countDistinct("order_id").alias("total_orders")
    )

# Apply segmentation logic
task6_result = customer_summary.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
).select(
    "customer_id",
    col("customer_city").alias("city"),
    "total_spend",
    "segment",
    "total_orders"
).orderBy(col("total_spend").desc())

print("=== Task 6: Final Reporting Table ===")
print(f"Output columns: customer_id, city, total_spend, segment, total_orders")
print(f"Total customers: {task6_result.count():,}\n")
display(task6_result.limit(50))

=== Task 6: Final Reporting Table ===
Output columns: customer_id, city, total_spend, segment, total_orders
Total customers: 98,666



customer_id,city,total_spend,segment,total_orders
1617b1357756262bfa56ab541c47bc16,rio de janeiro,13440.0,Gold,1
ec5b2ba62e574342386871631fafd3fc,vila velha,7160.0,Silver,1
c6e2731c5b391845f6800c97401a43a9,campo grande,6735.0,Silver,1
f48d464a0baaea338cb25f816991ab1f,vitoria,6729.0,Silver,1
3fd6777bbce08a352fddd04e4a7cc8f6,marilia,6499.0,Silver,1
05455dfa7cd02f13d132aa7a6a9729c6,divinopolis,5934.6,Silver,1
df55c14d1476a9a3467f131269c2477f,araruama,4799.0,Bronze,1
24bbf5fd2f2e1b359ee7de94defc4a15,maua,4690.0,Bronze,1
e0a2412720e9ea4f26c1ac985f6a7358,goiania,4599.9,Bronze,1
3d979689f636322c62418b6346b1c6d2,joao pessoa,4590.0,Bronze,1
